# New Evaluation Measure

ADAF implementation of new evaluation measures for archaeology, developed by Fiorucci et al. (2024).
 

In [1]:
import evaluate
import geopandas as gpd
from pathlib import Path
from shapely.geometry import MultiPolygon, Polygon
import numpy as np
# import copy
# import shapely.wkt

## Centroid-Based Measure 

In [2]:
pred_path = Path(r"c:\Users\ncoz\Desktop\stone_ml\adaf_irish.gpkg")


gt_path = Path(r"c:\Users\ncoz\Desktop\stone_ml\archaeology\gomile_2025-11-28.gpkg")
splti_pth = Path(r"c:\Users\ncoz\Desktop\stone_ml\ml_dataset_split_v2.gpkg")

In [3]:
# Load data

pred_gdf = gpd.read_file(pred_path)
gt_gdf = gpd.read_file(gt_path)
print("pred_gdf:", len(pred_gdf))
print("gt_gdf:", len(gt_gdf))

split_gdf = gpd.read_file(splti_pth)

pred_gdf: 11798
gt_gdf: 8052


In [4]:
pred_gdf["area"] = pred_gdf.geometry.area
pred_gdf["roundness"] = (4 * np.pi * pred_gdf.geometry.area / (pred_gdf.geometry.convex_hull.length ** 2)).round(3)

In [5]:
# (WILL NOT BE PART OF THE FUNCTION, MOVE OUT!)

# Post-processing filtering
before_len = len(pred_gdf)
print("Before filtering:", before_len)

pred_gdf = pred_gdf[pred_gdf["area"] < 1500].reset_index(drop=True)
pred_gdf = pred_gdf[(pred_gdf["area"] >= 10) & (pred_gdf["roundness"] > 0.7)].reset_index(drop=True)


print("After filtering:", len(pred_gdf))
print("Removed filtering:", before_len - len(pred_gdf))

Before filtering: 11798
After filtering: 11120
Removed filtering: 678


In [7]:
# (WILL NOT BE PART OF THE FUNCTION, MOVE OUT!)

# Filter by 'test' and convert into shaply multipolygon 

split_test = split_gdf[split_gdf['split'] == 'test'].copy()

# 3) Dissolve to one geometry (faster & avoids duplicates)
test_geom = split_test.unary_union

# 4) Spatial join against a single-row GeoDataFrame
test_one = gpd.GeoDataFrame({"geometry": [test_geom]}, crs=pred_gdf.crs)

pred_test = (
    gpd.sjoin(pred_gdf, test_one, how="inner", predicate="intersects")
        .drop(columns=["index_right"])
        .reset_index(drop=True)
)

gt_test = (
    gpd.sjoin(gt_gdf, test_one, how="inner", predicate="intersects")
        .drop(columns=["index_right"])
        .reset_index(drop=True)
)

print(len(pred_test))
print(len(gt_test))

891
1917


In [8]:
def gdf_to_multipolygon_nondissolving(gdf: gpd.GeoDataFrame) -> MultiPolygon:
    """
    Convert a GeoDataFrame to a single MultiPolygon without dissolving geometries.

    - Polygons are kept as-is
    - MultiPolygons are flattened into individual Polygons
    - Empty / None geometries are ignored
    """
    polygons = []

    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type == "Polygon":
            polygons.append(geom)
        elif geom.geom_type == "MultiPolygon":
            polygons.extend(list(geom.geoms))
        else:
            raise TypeError(f"Unsupported geometry type: {geom.geom_type}")

    return MultiPolygon(polygons)
    #return polygons


pred_test_mp = gdf_to_multipolygon_nondissolving(pred_test)
gt_test_mp   = gdf_to_multipolygon_nondissolving(gt_test)

print(gt_test_mp)

print("Predictions:", len(pred_test_mp.geoms))
print("Ground Truth:", len(gt_test_mp.geoms))

MULTIPOLYGON (((284194.51107117604 4763898.770857615, 284193.3387634737 4763900.789280602, 284193.6338966992 4763902.901648213, 284194.51130140317 4763904.919994462, 284196.76473948243 4763906.52312448, 284199.0468529564 4763906.880759454, 284201.3106695351 4763906.726619821, 284203.45432634914 4763905.25910225, 284204.42821264494 4763903.833406099, 284204.83028287545 4763902.794173916, 284204.43851449946 4763900.026424393, 284203.7673727105 4763897.6346834665, 284201.21626451175 4763895.895787416, 284198.1351615035 4763895.713125776, 284195.7695587226 4763897.115374115, 284194.51107117604 4763898.770857615)), ((284164.85473925143 4763912.861457161, 284164.507559666 4763915.436013723, 284164.44236744265 4763917.707672237, 284165.1944618863 4763920.31613182, 284166.79774599225 4763922.162118533, 284169.8997603768 4763922.929665071, 284175.2525223951 4763923.177518372, 284178.075086175 4763922.271361077, 284179.6155609058 4763920.312979198, 284180.1846862278 4763917.803691995, 284180.087

In [ ]:
pred_shapes = {"barrow": list(pred_test_mp.geoms)}
gt_shapes   = {"barrow": list(gt_test_mp.geoms)}

my_res = evaluate.compute_iou_metric_centroid(["barrow"], pred_shapes, gt_shapes, 0)


print("Total predictions:", len(pred_test))

my_res

{'barrow': {'TP': 742, 'FP': 149, 'FN': 1175}}